# Startup Pitch Evaluation - Colab Deployment with ngrok

This notebook deploys your FastAPI backend on Google Colab with T4 GPU and exposes it via ngrok.

**Setup Time:** ~5 minutes

**What you need:**
1. ngrok auth token from https://dashboard.ngrok.com/auth/your-authtoken
2. GitHub repo access (already cloned in this notebook)

## Cell 1: Check GPU & System Info

In [ ]:
import torch
import subprocess
import os

print("="*60)
print("SYSTEM INFO")
print("="*60)

# GPU Info
gpu_available = torch.cuda.is_available()
print(f"\n🎮 GPU Available: {gpu_available}")
if gpu_available:
    print(f"   Device: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"   CUDA Version: {torch.version.cuda}")
else:
    print("   ⚠️  WARNING: GPU not available. Go to Runtime → Change runtime type → T4 GPU")

# Python version
import sys
print(f"\n🐍 Python: {sys.version.split()[0]}")

# ffmpeg check
try:
    result = subprocess.run(['ffmpeg', '-version'], capture_output=True, text=True)
    print(f"\n🎬 ffmpeg: ✅ Installed")
except:
    print(f"\n🎬 ffmpeg: ❌ Not found (will install)")

## Cell 2: Install System Dependencies

In [ ]:
# Update and install system packages
!apt-get update -qq
!apt-get install -y -qq ffmpeg libsm6 libxext6 libxrender-dev > /dev/null 2>&1

print("✅ System dependencies installed")

## Cell 3: Install Python Dependencies

In [ ]:
# Install all Python packages
packages = [
    # API & Web
    "fastapi==0.116.1",
    "uvicorn==0.35.0",
    "pydantic==2.11.7",
    "pydantic-settings==2.10.1",
    "python-multipart==0.0.20",
    
    # ML & CV
    "numpy==2.2.6",
    "opencv-python-headless==4.10.0.84",
    "mediapipe==0.10.14",
    "imageio-ffmpeg==0.6.0",
    
    # Audio & NLP
    "faster-whisper==1.1.1",
    "sentence-transformers>=2.7",
    "langdetect==1.0.9",
    
    # Deep Learning
    "torch>=2.0",
    "torchaudio>=2.0",
    "torchvision>=0.15",
    
    # Tunneling
    "pyngrok==7.2.1",
    
    # Testing
    "pytest==8.4.1",
    "httpx==0.28.1",
]

print("Installing Python packages...")
for package in packages:
    print(f"  📦 {package}", end="... ")
    !pip install -q {package} 2>/dev/null
    print("✅")

print("\n✅ All Python dependencies installed")

## Cell 4: Clone Repository

In [ ]:
import requests
import json

# Load API URL
try:
    with open('/tmp/colab_api_config.json', 'r') as f:
        config = json.load(f)
        api_url = config['public_url']
except Exception:
    print("❌ Run Cell 7 first to start the API.")
    api_url = None

if api_url:
    print(f"Testing Evaluation Endpoint\n")

    payload = {
        "title": "TechFlow - AI Sales Automation",
        "transcript": (
            "Hello, I'm the CEO of TechFlow. We've built an AI-powered sales automation "
            "platform that helps enterprise sales teams close deals 2x faster. "
            "Our AI assistant handles lead qualification, follow-ups, and proposal generation. "
            "We've already onboarded 150 companies across North America and Europe, "
            "generating $4.5M ARR with a growth rate of 25% MoM. "
            "Our team consists of 12 engineers from companies like Salesforce and HubSpot. "
            "We're seeking $5M Series A to expand into APAC and build additional AI features."
        ),
        "language_hint": "en",
        "slide_text": [
            "Problem: Sales reps waste 60% of time on manual tasks",
            "Solution: AI handles qualification, follow-ups, proposals",
            "150+ Enterprise Customers",
            "$4.5M ARR, 25% MoM Growth",
            "Team: Ex-Salesforce, HubSpot Engineers",
            "Ask: $5M Series A",
        ],
        "presenter_profile": {
            "role": "CEO",
            "experience_years": 8,
            "background": "MIT, previously at Salesforce",
        },
        "user_details": {},
    }

    print("Sending request...\n")

    try:
        response = requests.post(
            f"{api_url}/evaluate",
            json=payload,
            timeout=120,
        )

        if response.status_code == 200:
            result = response.json()
            summary = result.get("summary", {})

            print("✅ Evaluation Successful\n")
            print(f"{'='*60}")
            print(f"RESULTS")
            print(f"{'='*60}")
            print(f"\n📊 Overall Score: {summary.get('overall_score', 0):.1f}/10")
            print(f"💰 Investment Band: {summary.get('investment_band', 'unknown')}")
            print(f"🆔 Request ID: {result.get('request_id', 'unknown')}")

            print(f"\n📈 Rubric/Dashboard Data:")
            dashboard = result.get('dashboard', {})
            for name, values in {
                "quantitative_scores": dashboard.get('quantitative_scores', []),
                "modality_weights": dashboard.get('modality_weights', []),
                "risk_distribution": dashboard.get('risk_distribution', []),
            }.items():
                print(f"   • {name}: {len(values)} items")

            print(f"\n💪 Top Strengths:")
            for i, strength in enumerate(summary.get('strengths', [])[:3], 1):
                print(f"   {i}. {strength}")

            print(f"\n⚠️  Key Weaknesses:")
            for i, weakness in enumerate(summary.get('weaknesses', [])[:3], 1):
                print(f"   {i}. {weakness}")

            print(f"\n💡 Top Suggestions:")
            for i, suggestion in enumerate(summary.get('suggestions', [])[:3], 1):
                print(f"   {i}. {suggestion}")

            print(f"\n{'='*60}")
            print("✅ API is working perfectly!")
            print("\nFull result stored as: result = response.json()")
        else:
            print(f"❌ API Error: {response.status_code}")
            print(response.text)

    except Exception as e:
        print(f"❌ Request failed: {e}")


In [ ]:
import os
from pathlib import Path

backend_dir = Path("/content/pitch-eval/backend")
env_path = backend_dir / ".env"

env_text = """SPE_USE_HEURISTIC_PIPELINE=false
SPE_USE_LOCAL_TRANSCRIBER=true
SPE_ENABLE_VISUAL_EXTRACTION=true
SPE_ENABLE_AUDIO_EXTRACTION=true
SPE_TRANSCRIBER_BACKEND=auto
SPE_TRANSCRIBER_MIN_AUDIO_QUALITY=0.35
SPE_OPENAI_API_KEY=
SPE_OPENAI_TRANSCRIBER_MODEL=whisper-1
SPE_FASTER_WHISPER_MODEL_SIZE=small
SPE_FASTER_WHISPER_DEVICE=cuda
SPE_FASTER_WHISPER_COMPUTE_TYPE=float16
SPE_MEDIA_LOOKUP_DIR=outputs/batch_input
SPE_NN_CHECKPOINT_PATH=models/checkpoints/phase6_gpu_nn_model.pt
SPE_NN_TEXT_ENCODER=all-MiniLM-L6-v2
SPE_NN_VISUAL_BACKBONE=mobilenet_v3_small
SPE_NN_AUDIO_FEATURES=mfcc
SPE_NN_DEVICE=auto
"""

env_path.write_text(env_text, encoding="utf-8")
print(f"✅ Wrote neural-mode env file: {env_path}")
print("   Heuristic mode is disabled and the phase6 checkpoint is selected.")


## Cell 5: Configure ngrok Auth Token

In [ ]:
import os
from getpass import getpass

print("="*60)
print("NGROK SETUP")
print("="*60)
print("\n1. Get your auth token from: https://dashboard.ngrok.com/auth/your-authtoken")
print("2. Paste it below (will be hidden)\n")

ngrok_token = getpass("Enter your ngrok auth token: ")

if ngrok_token.strip():
    os.environ['NGROK_AUTHTOKEN'] = ngrok_token
    
    # Also configure ngrok via CLI
    !ngrok config add-authtoken {ngrok_token}
    
    print("\n✅ ngrok auth token configured")
else:
    print("\n❌ No token provided. ngrok tunneling will not work.")

## Cell 6: Verify Model Checkpoint

In [ ]:
from pathlib import Path
import os

checkpoint_path = Path("/content/pitch-eval/backend/models/checkpoints/phase6_gpu_nn_model.pt")
config_path = Path("/content/pitch-eval/backend/models/config/training_gpu.yaml")

print("Checking model files...\n")

if checkpoint_path.exists():
    size_mb = checkpoint_path.stat().st_size / (1024**2)
    print(f"✅ Model checkpoint: {checkpoint_path.name} ({size_mb:.1f} MB)")
else:
    print(f"❌ Model checkpoint NOT found: {checkpoint_path}")

if config_path.exists():
    print(f"✅ GPU config: {config_path.name}")
else:
    print(f"⚠️  GPU config NOT found: {config_path}")

## Cell 7: Start FastAPI Server with ngrok Tunnel

In [ ]:
import subprocess
import threading
import time
import os
from pathlib import Path

# Global variables to track server
server_process = None
public_url = None

def start_server():
    """Start FastAPI server in background"""
    global server_process
    
    backend_dir = Path("/content/pitch-eval/backend")
    os.chdir(backend_dir)
    
    print(f"Starting FastAPI server from: {backend_dir}")
    
    server_process = subprocess.Popen(
        [
            "uvicorn",
            "app.main:app",
            "--host", "0.0.0.0",
            "--port", "8000",
            "--workers", "2",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    print("✅ FastAPI server started (PID: {})\n".format(server_process.pid))

def create_ngrok_tunnel():
    """Create ngrok tunnel and return public URL"""
    global public_url
    
    try:
        from pyngrok import ngrok
        
        # Give server time to start
        print("Waiting for server to start...")
        time.sleep(3)
        
        # Create tunnel
        print("Creating ngrok tunnel...")
        tunnel = ngrok.connect(8000, "http", bind_tls=True)
        public_url = tunnel.public_url
        
        print(f"\n{'='*60}")
        print(f"✅ API IS LIVE AND ACCESSIBLE")
        print(f"{'='*60}")
        print(f"\n🌐 Public URL: {public_url}")
        print(f"\n📍 Endpoints:")
        print(f"   • Health:    {public_url}/health")
        print(f"   • Evaluate:  POST {public_url}/evaluate")
        print(f"   • Batch:     POST {public_url}/evaluate/batch")
        print(f"   • Frontend:  {public_url}/")
        print(f"\n{'='*60}\n")
        
        return public_url
        
    except Exception as e:
        print(f"\n❌ ngrok tunnel failed: {e}")
        print("   Make sure you configured ngrok auth token in Cell 5")
        return None

# Start server
start_server()

# Create tunnel
public_url = create_ngrok_tunnel()

# Store in notebook for later use
import json
with open('/tmp/colab_api_config.json', 'w') as f:
    json.dump({'public_url': public_url}, f)

print("\n⏳ Server is running. Proceed to next cells to test.")

## Cell 8: Test API Health Check

In [ ]:
import requests
import json
import time

# Load config from previous cell
try:
    with open('/tmp/colab_api_config.json', 'r') as f:
        config = json.load(f)
        api_url = config['public_url']
except:
    print("❌ Config not found. Run Cell 7 first.")
    api_url = None

if api_url:
    print(f"Testing API: {api_url}\n")
    
    # Try health check with retries
    max_retries = 5
    for attempt in range(max_retries):
        try:
            response = requests.get(f"{api_url}/health", timeout=5)
            
            if response.status_code == 200:
                data = response.json()
                print("✅ Health Check: SUCCESS\n")
                print(json.dumps(data, indent=2))
                break
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                print(f"⏳ Retry {attempt+1}/{max_retries}... (server starting up)")
                time.sleep(2)
            else:
                print(f"❌ Health Check Failed: {e}")
                print("   The server may still be initializing. Wait 30 seconds and retry.")

## Cell 9: Test Single Evaluation

In [ ]:
import requests
import json

# Load API URL
try:
    with open('/tmp/colab_api_config.json', 'r') as f:
        config = json.load(f)
        api_url = config['public_url']
except:
    print("❌ Run Cell 7 first to start the API.")
    api_url = None

if api_url:
    print(f"Testing Evaluation Endpoint\n")
    
    payload = {
        "title": "TechFlow - AI Sales Automation",
        "transcript": (
            "Hello, I'm the CEO of TechFlow. We've built an AI-powered sales automation "
            "platform that helps enterprise sales teams close deals 2x faster. "
            "Our AI assistant handles lead qualification, follow-ups, and proposal generation. "
            "We've already onboarded 150 companies across North America and Europe, "
            "generating $4.5M ARR with a growth rate of 25% MoM. "
            "Our team consists of 12 engineers from companies like Salesforce and HubSpot. "
            "We're seeking $5M Series A to expand into APAC and build additional AI features."
        ),
        "language_hint": "en",
        "slide_text": [
            "Problem: Sales reps waste 60% of time on manual tasks",
            "Solution: AI handles qualification, follow-ups, proposals",
            "150+ Enterprise Customers",
            "$4.5M ARR, 25% MoM Growth",
            "Team: Ex-Salesforce, HubSpot Engineers",
            "Ask: $5M Series A"
        ],
        "presenter_profile": {
            "role": "CEO",
            "experience_years": 8,
            "background": "MIT, previously at Salesforce"
        },
        "user_details": {}
    }
    
    print("Sending request...\n")
    
    try:
        response = requests.post(
            f"{api_url}/evaluate",
            json=payload,
            timeout=120
        )
        
        if response.status_code == 200:
            result = response.json()
            
            print(f"✅ Evaluation Successful\n")
            print(f"{'='*60}")
            print(f"RESULTS")
            print(f"{'='*60}")
            print(f"\n📊 Overall Score: {result['overall_score']:.1f}/100")
            print(f"💰 Investment Band: {result['investment_band']}")
            print(f"🆔 Request ID: {result['request_id']}")
            
            print(f"\n📈 Rubric Scores:")
            rubric = result.get('rubric_scores', {})
            for metric_name, score in rubric.items():
                bar = "█" * int(score) + "░" * (10 - int(score))
                print(f"   {metric_name:25s} {bar} {score:.1f}/10")
            
            print(f"\n💪 Top Strengths:")
            for i, strength in enumerate(result.get('strengths', [])[:3], 1):
                print(f"   {i}. {strength}")
            
            print(f"\n⚠️  Key Weaknesses:")
            for i, weakness in enumerate(result.get('weaknesses', [])[:3], 1):
                print(f"   {i}. {weakness}")
            
            print(f"\n💡 Top Suggestions:")
            for i, suggestion in enumerate(result.get('suggestions', [])[:3], 1):
                print(f"   {i}. {suggestion}")
            
            print(f"\n{'='*60}")
            print(f"\n✅ API is working perfectly!")
            print(f"\nFull result stored as: result = response.json()")
            
        else:
            print(f"❌ API Error: {response.status_code}")
            print(response.text)
            
    except Exception as e:
        print(f"❌ Request failed: {e}")

## Cell 10: Get Your Public API URL

In [ ]:
import json

try:
    with open('/tmp/colab_api_config.json', 'r') as f:
        config = json.load(f)
        api_url = config['public_url']
        
        print("\n" + "="*70)
        print("YOUR API IS LIVE AND READY TO USE")
        print("="*70)
        print(f"\n🌐 Base URL: {api_url}")
        print(f"\n📝 Use this URL to:")
        print(f"   1. Deploy frontend on Vercel/GitHub Pages")
        print(f"   2. Test with curl:")
        print(f"      curl -X GET {api_url}/health")
        print(f"\n   3. Test with Python:")
        print(f"      import requests")
        print(f"      requests.post('{api_url}/evaluate', json=payload)")
        print(f"\n📌 IMPORTANT: Keep this Colab notebook open!")
        print(f"   • Closing it will stop the API")
        print(f"   • ngrok URL may change if connection drops")
        print(f"\n" + "="*70 + "\n")
except:
    print("❌ Config not found. Run cells 7-9 first.")

## Cell 11: Keep Server Alive (Run This Continuously)

In [ ]:
import time
import json
import requests
from datetime import datetime

print("\n🔄 Server Monitor Started\n")
print("This cell will keep the API alive and show status updates.")
print("Let it run continuously in the background.\n")
print("="*70)

try:
    with open('/tmp/colab_api_config.json', 'r') as f:
        config = json.load(f)
        api_url = config['public_url']
except:
    print("❌ Config not found")
    api_url = None

if api_url:
    request_count = 0
    error_count = 0
    
    while True:
        try:
            # Health check every 30 seconds
            response = requests.get(f"{api_url}/health", timeout=5)
            
            if response.status_code == 200:
                timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
                print(f"[{timestamp}] ✅ API Status: HEALTHY")
                error_count = 0
            else:
                error_count += 1
                print(f"⚠️  Status: {response.status_code} (Error count: {error_count})")
                
        except requests.exceptions.RequestException as e:
            error_count += 1
            print(f"❌ Connection Error: {str(e)[:50]} (Error count: {error_count})")
        
        # Sleep for 30 seconds before next check
        time.sleep(30)

## Cell 12: Stop Server (If Needed)

In [ ]:
import subprocess
import os
import signal

# Kill all uvicorn processes
!pkill -f uvicorn

print("✅ FastAPI server stopped")

---

## 🚀 Quick Start Guide

### Step 1: Get ngrok token
1. Go to https://dashboard.ngrok.com/auth/your-authtoken
2. Copy your auth token

### Step 2: Run cells in order
1. **Cell 1:** Check GPU (should say T4 GPU is available)
2. **Cell 2-3:** Install dependencies
3. **Cell 4:** Clone repo
4. **Cell 5:** Paste ngrok token
5. **Cell 6:** Verify model checkpoint
6. **Cell 7:** Start API server + ngrok tunnel
7. **Cell 8:** Test health check
8. **Cell 9:** Test evaluation
9. **Cell 10:** Get your public URL
10. **Cell 11:** Keep server alive (run continuously)

### Step 3: Use your API
- Copy the URL from Cell 10
- Deploy frontend (Vercel/GitHub Pages)
- Frontend makes API calls to that URL

### Step 4: Keep running
- **DO NOT close this notebook**
- **DO NOT disconnect from Colab**
- Let Cell 11 run continuously
- Your API will be live 24/7 (until Colab cuts you off)

---

## 📌 Important Notes

- **Colab Runtime Limit:** Free tier = 12 hours per session
- **URL Persistence:** URL changes if connection drops (use ngrok paid tier for fixed URL)
- **Concurrent Users:** ~10-20 light users on free ngrok tier
- **GPU Memory:** ~15GB available, your model uses ~2-4GB
- **Bandwidth:** ~1GB/month on free ngrok (upgrade to paid for unlimited)

---